# 03. 상관 분석 (입력 변수 상관행렬, 수온과 출현율의 관계)
- 대상: 2014-09 ~ 2023-12 ('추가(2024)' 20행 제외), 입력 변수 11개
- 상관분석은 변수 선택에 쓰지 않고 관계를 설명하는 용도라 전체 기간 사용 (변수 제거 검토는 04에서 train으로만)
- 센서 고장값(포항 부이 2018-12-06~16 수온)은 DB 전처리에서 결측 처리됨 (공통 셀에서 확인)

## 공통 설정·전처리
01_모델비교와 같은 설정과 함수예요. 이 노트북만 따로 실행해도 되도록 그대로 넣었어요.

In [1]:
import itertools
import os
import warnings

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, precision_score,
                             recall_score, roc_auc_score)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

In [2]:
# ============================== [설정] ==============================
DATA_PATH = "학습데이터.csv"      # 최종 학습데이터 파일 (.csv 또는 .xlsx, 열 이름은 v_학습데이터 뷰와 같아야 함)
SHEET_NAME = 0                    # .xlsx일 때 읽을 시트 이름 또는 번호
OUT_DIR = "결과"

# 입력 변수 11개 (제안서 3.6절). 수온_변화_1주만 코드에서 만들고 나머지는 파일의 열 그대로 사용
FEATURES = [
    "통합", "통합_1주전", "통합_2주전", "통합_변화_1주",   # 현재·1~2주 전 출현율과 변화량
    "연중주차",                                          # 계절
    "수온_주평균", "수온_평년차", "수온_변화_1주",          # 수온·수온 평년차·수온 변화량
    "기온_주평균", "풍속_주평균", "유의파고_주평균",         # 기온·풍속·파고
]
BASELINE_FEATURES = ["통합"]      # 기준선: 현재 출현율만 사용 (비교용)

HORIZONS = [1, 2, 4]              # 1·2·4주 뒤 예측
TARGET = "타깃_급증_{k}주후"        # 정답 열 이름 형식
DATE_COL = "시작일"                # 시간 순서 분할 기준 열
SPLIT_RATIO = (0.7, 0.1, 0.2)     # train : valid : test
EXCLUDE_SPLIT = ["추가(2024)"]     # '분할' 열에서 이 값인 행은 제외 (2024년 5월 보고 종료로 생긴 짧은 구간)
THRESH = 0.5                      # 재현율·정밀도 계산용 확률 임계값
SEED = 42

# 데이터 오류는 DB 전처리(전처리_보정.py)에서만 고치고, 여기서는 고쳐졌는지 확인만 한다
#   포항 부이(22106) 2018-12-06~16 수온 센서 고장 → DB에서 해당 주 수온 NULL
#   → 이 기간 주평균을 쓰는 경북 2018-11-30, 2018-12-07 보고의 수온이 비어 있어야 함
SENSOR_CHECK = [{"관측소ID": 22106, "시작일": ["2018-11-30", "2018-12-07"]}]
TEMP_COLS = ["수온_주평균", "수온_평년차"]
os.makedirs(OUT_DIR, exist_ok=True)

In [3]:
def load_data(path=DATA_PATH):
    """파일을 읽고 필요한 열을 확인한 뒤, '추가(2024)' 행을 빼고 권역·날짜 순으로 정렬한다."""
    if path.lower().endswith((".xlsx", ".xls")):
        df = pd.read_excel(path, sheet_name=SHEET_NAME)
    else:
        try:
            df = pd.read_csv(path, encoding="utf-8-sig")
        except UnicodeDecodeError:
            df = pd.read_csv(path, encoding="cp949")
    df.columns = df.columns.str.strip()

    base_cols = [c for c in FEATURES if c != "수온_변화_1주"]
    need = ["권역명", DATE_COL] + base_cols + [TARGET.format(k=k) for k in HORIZONS]
    missing = [c for c in need if c not in df.columns]
    if missing:
        raise ValueError(f"파일에 다음 열이 없습니다: {missing}")

    if "분할" in df.columns and EXCLUDE_SPLIT:
        df = df[~df["분할"].isin(EXCLUDE_SPLIT)]
    df[DATE_COL] = pd.to_datetime(df[DATE_COL])
    df = df.sort_values(["권역명", DATE_COL]).reset_index(drop=True)
    check_sensor_errors(df)
    return df


def check_sensor_errors(df):
    """DB에서 센서 고장값이 결측 처리됐는지 확인만 한다 (값은 바꾸지 않음)."""
    for e in SENSOR_CHECK:
        m = (df["관측소ID"] == e["관측소ID"]) & df[DATE_COL].isin(pd.to_datetime(e["시작일"]))
        left = df.loc[m, TEMP_COLS].notna().any(axis=1).sum()
        if left:
            print(f"⚠️ 경고: 관측소 {e['관측소ID']} 센서 고장 구간 수온이 {left}행 남아 있음 → DB 전처리(전처리_보정.py) 확인 필요")
        else:
            print(f"확인: 관측소 {e['관측소ID']} 센서 고장 구간 수온 결측 처리됨 ({m.sum()}행)")


def add_features(df):
    """제안서의 '수온 변화량' = 이번 주 수온 − 같은 권역 직전 주 수온.
    직전 주의 기준은 v_학습데이터 뷰의 통합_1주전과 똑같이 맞춤 (같은 권역의 4~10일 전 보고가 있을 때만 계산, 없으면 결측).
    현재·과거 값만 쓰므로 데이터 누수 없음."""
    prev_date = df.groupby("권역명")[DATE_COL].shift()
    prev_temp = df.groupby("권역명")["수온_주평균"].shift()
    gap = (df[DATE_COL] - prev_date).dt.days
    df["수온_변화_1주"] = np.where(gap.between(4, 10), df["수온_주평균"] - prev_temp, np.nan)
    return df


def time_split(df):
    """시작일 순서로 7:1:2 분할. 같은 주(시작일)의 권역들은 같은 세트에 들어가게 날짜 단위로 자른다."""
    counts = df[DATE_COL].value_counts().sort_index()
    cum = df[DATE_COL].map(counts.cumsum()) / len(df)   # 그 날짜까지의 누적 비율
    tr, va, _ = SPLIT_RATIO
    return np.where(cum <= tr, "train", np.where(cum <= tr + va, "valid", "test"))


# ---------- 모델 정의 ----------
def logistic(pos_weight, C):
    # 결측 대체(중앙값)·표준화 기준은 fit 때 train으로만 계산되고, valid·test에는 그대로 적용됨
    return make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(C=C, class_weight="balanced", max_iter=2000),
    )


def lightgbm(pos_weight, **p):
    # 결측(NaN)은 그대로 넣음: 분기마다 결측을 어느 쪽으로 보낼지 학습함
    return LGBMClassifier(class_weight="balanced", subsample=0.8, subsample_freq=1,
                          colsample_bytree=0.8, random_state=SEED, verbose=-1, **p)


def xgboost(pos_weight, **p):
    # 결측(NaN)은 그대로 넣음. 클래스 불균형은 scale_pos_weight = 음성 수 / 양성 수 (train 기준)
    return XGBClassifier(scale_pos_weight=pos_weight, subsample=0.8, colsample_bytree=0.8,
                         eval_metric="logloss", random_state=SEED, n_jobs=-1, **p)


def grid(**kw):
    return [dict(zip(kw, v)) for v in itertools.product(*kw.values())]


# 이름: (사용 변수, 모델 생성 함수, 하이퍼파라미터 후보, 최종 선정 대상 여부)
MODELS = {
    "기준선":   (BASELINE_FEATURES, logistic, [{"C": 1.0}], False),
    "Logistic": (FEATURES, logistic, grid(C=[0.01, 0.03, 0.1, 0.3, 1, 3]), True),
    "LightGBM": (FEATURES, lightgbm, grid(n_estimators=[100, 300], learning_rate=[0.03, 0.1],
                                          num_leaves=[7, 15], min_child_samples=[10, 30]), True),
    "XGBoost":  (FEATURES, xgboost, grid(n_estimators=[100, 300], learning_rate=[0.03, 0.1],
                                         max_depth=[3, 5], min_child_weight=[1, 5]), True),
}


def scores(y, p):
    pred = (p >= THRESH).astype(int)
    return {"AUC": roc_auc_score(y, p), "PR_AUC": average_precision_score(y, p),
            "재현율": recall_score(y, pred), "정밀도": precision_score(y, pred, zero_division=0)}


# 데이터 준비 (전처리 + 분할)
df = add_features(load_data())
df["세트"] = time_split(df)
print(f"데이터 {len(df)}행 / 입력 변수 {len(FEATURES)}개")

확인: 관측소 22106 센서 고장 구간 수온 결측 처리됨 (2행)
데이터 2534행 / 입력 변수 11개


## 1. 각 변수와 통합 출현율(피어슨·스피어만), 1·2·4주 뒤 급증 여부(점이연 상관)

In [4]:
rows = []
for f in FEATURES:
    r = {"변수": f,
         "출현율_피어슨": df[f].corr(df["통합"]),
         "출현율_스피어만": df[f].corr(df["통합"], method="spearman")}
    for k in HORIZONS:
        y = TARGET.format(k=k)
        d = df.dropna(subset=[y])
        r[f"급증_{k}주뒤"] = d[f].corr(d[y])          # 0/1 정답과의 피어슨 = 점이연 상관
    rows.append(r)
tgt = pd.DataFrame(rows).round(3)
tgt

,변수,출현율_피어슨,출현율_스피어만,급증_1주뒤,급증_2주뒤,급증_4주뒤
0,통합,1.000,1.000,0.244,0.231,0.137
1,통합_1주전,0.756,0.749,0.304,0.245,0.115
2,통합_2주전,0.683,0.675,0.212,0.166,0.029
3,통합_변화_1주,0.342,0.306,-0.128,-0.065,-0.020
4,연중주차,-0.326,-0.367,-0.177,-0.254,-0.360
5,수온_주평균,0.464,0.480,0.146,0.085,-0.033
6,수온_평년차,0.164,0.166,0.056,0.095,0.150
7,수온_변화_1주,0.220,0.242,0.103,0.159,0.248
8,기온_주평균,0.495,0.515,0.181,0.163,0.092
9,풍속_주평균,-0.122,-0.144,-0.068,-0.121,-0.115


## 2. 입력 변수끼리의 상관 (|r| ≥ 0.7)

In [5]:
inp = df[FEATURES].corr().round(3)
high = [(a, b, inp.loc[a, b]) for i, a in enumerate(FEATURES) for b in FEATURES[i + 1:] if abs(inp.loc[a, b]) >= 0.7]
pd.DataFrame(sorted(high, key=lambda x: -abs(x[2])), columns=["변수1", "변수2", "상관계수"])

,변수1,변수2,상관계수
0,수온_주평균,기온_주평균,0.894
1,풍속_주평균,유의파고_주평균,0.782
2,통합_1주전,통합_2주전,0.758
3,통합,통합_1주전,0.756


## 3. 수온 구간별 평균 출현율, 같은 달 안에서의 상관 (계절 효과 제거)

In [6]:
TEMP_BINS = [0, 12, 16, 20, 24, 28, 32]
bins = (df.groupby(pd.cut(df["수온_주평균"], TEMP_BINS), observed=True)["통합"]
          .agg(평균출현율="mean", 행수="size").round(1))
bins.index = [f"{int(i.left)}~{int(i.right)}℃" for i in bins.index]
display(bins)

month = df[DATE_COL].dt.month
within = pd.DataFrame({f: df.groupby(month).apply(lambda g: g[f].corr(g["통합"], method="spearman"))
                       for f in ["수온_주평균", "수온_평년차"]})
within["행수"] = df.groupby(month).size()
within.index.name = "월"
within = within.round(3)
within

,평균출현율,행수
0~12℃,4.6,18
12~16℃,13.4,192
16~20℃,21.6,667
20~24℃,36.0,953
24~28℃,48.0,626
28~32℃,50.1,73


,수온_주평균,수온_평년차,행수
월,,,
5,0.330,0.068,149
6,0.242,0.199,308
7,0.322,0.158,382
8,-0.012,-0.103,384
9,0.108,0.034,475
10,0.127,0.046,445
11,0.117,0.034,347
12,0.186,-0.225,44


## 4. 저장

In [7]:
inp.to_csv(f"{OUT_DIR}/상관_입력변수.csv", encoding="utf-8-sig")
tgt.to_csv(f"{OUT_DIR}/상관_출현율_급증.csv", index=False, encoding="utf-8-sig")
within.to_csv(f"{OUT_DIR}/상관_수온_월별.csv", encoding="utf-8-sig")
bins.to_csv(f"{OUT_DIR}/수온구간별_평균출현율.csv", encoding="utf-8-sig")
print(f"저장: {OUT_DIR}/상관_*.csv, 수온구간별_평균출현율.csv")

저장: 결과/상관_*.csv, 수온구간별_평균출현율.csv


## 결론
- 수온과 통합 출현율: 피어슨 0.46 (중간 정도 양의 상관). 수온 구간별 평균 출현율 12℃ 이하 5% → 20~24℃ 36% → 24℃ 이상 약 48~50%
- 수온 평년차와 출현율: 0.16 (약함)
- 같은 달 안에서는 상관이 −0.01~0.33으로 약해짐 → 관계의 상당 부분이 계절(여름) 효과. 인과관계로 해석하지 않음
- 입력 변수끼리 강한 상관: 수온–기온 0.89, 풍속–파고 0.78, 통합–1주 전 0.76, 1주 전–2주 전 0.76 → 04에서 변수 제거 검토 (상관행렬 그림은 04에서 제거 검토 결과와 함께 그림)